In [1]:
!pip install -q datasets transformers accelerate transformers[sentencepiece] sacrebleu rouge_score py7zr

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.3/71.3 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 495.3/495.3 kB 14.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.6/100.6 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.5/51.5 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 144.3/144.3 kB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 87.7 MB/s eta 0:00:00:00:0100:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]

**📦 1. Import Libraries & Suppress Warnings**

In [2]:
from datasets import load_dataset
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from transformers import TrainingArguments, Trainer
from transformers import pipeline
from transformers import DataCollatorForSeq2Seq
import warnings
warnings.filterwarnings("ignore")

**🤖 2. Load Model & Tokenize**

In [3]:
model_checkpoint="t5-small"
tokenizer= AutoTokenizer.from_pretrained(model_checkpoint)
model= AutoModelForSeq2SeqLM.from_pretrained(model_checkpoint).to("cuda")
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/242M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

True
Tesla T4


**📚 3. Load SAMSum Dataset**

In [4]:
dataset= load_dataset("knkarthick/samsum") 

README.md: 0.00B [00:00, ?B/s]

train.csv: 0.00B [00:00, ?B/s]

validation.csv: 0.00B [00:00, ?B/s]

test.csv: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/14731 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/818 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/819 [00:00<?, ? examples/s]

**✂️ 4. Tokenize the Dataset**

In [5]:
def tokenize_content(data):
    dialogues = data["dialogue"]
    summaries = data["summary"]

    inputs = [
        "summarize: " + text if text else "summarize:"
        for text in dialogues
    ]

    targets = [
        text if text else ""
        for text in summaries
    ]

    input_encoding = tokenizer(
        inputs,
        max_length=1024,
        truncation=True,
        padding="max_length"
    )

    target_encoding = tokenizer(
        text_target=targets,
        max_length=1024,
        truncation=True,
        padding="max_length"
    )

    return {
        "input_ids": input_encoding["input_ids"],
        "attention_mask": input_encoding["attention_mask"],
        "labels": target_encoding["input_ids"]
    }


tokenized_dataset = dataset.map(
    tokenize_content,
    batched=True
)
tokenized_dataset

Map:   0%|          | 0/14731 [00:00<?, ? examples/s]

Map:   0%|          | 0/818 [00:00<?, ? examples/s]

Map:   0%|          | 0/819 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['id', 'dialogue', 'summary', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 14731
    })
    validation: Dataset({
        features: ['id', 'dialogue', 'summary', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 818
    })
    test: Dataset({
        features: ['id', 'dialogue', 'summary', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 819
    })
})

**🧱 5. Setup Data Collator**

In [6]:
seq2seq_collator=DataCollatorForSeq2Seq(tokenizer,model=model)

**⚙️ 6. Define Training Arguments**

In [7]:
Training_args=TrainingArguments(
    output_dir= "t5-small-model",
    num_train_epochs=1,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    warmup_steps=500,
    weight_decay=0.01,
    logging_steps=10,
    eval_steps=500,
    save_steps=1e6,
    gradient_accumulation_steps=16,
    report_to='none'
)

**🏋️ 7. Initialize Trainer**

In [8]:
from transformers import Seq2SeqTrainer

trainer=Trainer(
    model=model,
    args=Training_args,
    data_collator=seq2seq_collator,
    train_dataset=tokenized_dataset['train'],
    eval_dataset=tokenized_dataset['validation'],
)

**🚀 8. Train the Model**

In [ ]:
trainer.train()

Step,Training Loss
10,462.883545
20,452.093115
30,443.558398
40,429.367822
50,401.505005
60,378.662085
70,341.478833
80,289.694312
90,245.446167
100,192.292603


**💾 9. Save Model & Tokenizer**

In [ ]:
model.save_pretrained("t5-finetuned-model")
tokenizer.save_pretrained("t5-samsum-tokenizer")

**🔁 10. Reload & Setup for Inference**

In [ ]:
tokenizer= AutoTokenizer.from_pretrained("t5-samsum-tokenizer")
model=AutoModelForSeq2SeqLM.from_pretrained("t5-finetuned-model").to("cuda")
summarizer=pipeline("summarization",model=model,tokenizer=tokenizer)

**🎭 11. Test Sample Dialogue (Luffy & Naruto)**

In [ ]:
test_text="""John: Hi Sarah, have you completed the sales report for this month?

Sarah: I'm almost done. I just need to verify the numbers from the west region.

John: Great. We need to present it to the management team tomorrow morning.

Sarah: I know. I'll finish the verification by this evening and send you the final report.

John: Perfect. Also, don't forget to include the comparison with last month's performance.

Sarah: Thanks for the reminder. I'll add that section as well.

John: Let me know if you need any help.

Sarah: Sure. I'll share the report once it's ready."""

**📄 12. Show the Summary Output**

In [ ]:
from Ipython.display import Markdown, display

result=summarizer(test_text,max_lenght=100,min_length=30,do_sample=False,model,tokenizer)

display(Markdown(f"***Summary:***"{result[0]["summary_test"]}))